# 09. Train matched predictors without information leakage

We will follow the six predictor heads, prove what each head may see, run a small normal-only training exercise, and inspect the exact same masked targets.

## Run this notebook on its own

From `experiments/aaai`, run `uv sync`, then `uv run jupyter lab`. Select the Python environment created here. Python 3.12, PyTorch, pandas, scikit-learn, matplotlib, python-dotenv, the notebook libraries, and the local `ambient` package are declared in `pyproject.toml`. Each notebook imports and loads everything it needs.

All configuration and credentials remain in the repository root `.env`. Set `AAAI_DEMO=1` for artificial arrays used to check the software, or `AAAI_DEMO=0` for available real poses. Restart the kernel after changing settings. Artificial examples have no medical meaning. The setup uses aliases for python-dotenv's actual function names so that `load_env(find_env())` works.

The video below provides real walking context in either mode. It is separate from any artificial example. These research notebooks do not launch a large download or full experiment without an explicit command.

## Historical tutorial and current study

This notebook preserves the earlier tutorial's code, observations and experimental scope. Its original result identities remain historical; they are not measurements of the new retained-predictor or U/J/A study. Start the current independent tutorial sequence at [Notebook 12](12_observation_population_and_video_context.ipynb), then continue through [Notebook 17](17_failure_analysis_and_manuscript_decision.ipynb). The [current protocol](docs/iteration02/protocol.md) and [current results](docs/iteration02/results.md) identify the verified new run. The [draft01 manuscript](docs/draft01/manuscript.md) remains the record of the completed earlier study.

The current primary question is normal versus the four requested condition families, with all other folders retained in a mandatory broad secondary analysis. It tests opposite-leg information beyond own-leg geometry and extra capacity. Uniform masking remains the reference, while J and A are declared motion-independent alternatives with matched budgets. Read the newer notebooks for their support rules and fixed evaluation banks. Do not combine old and new scores or denominators.

In [ ]:
from pathlib import Path
import os, sys, json
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'ambient').is_dir() and (p/'data/gavd').is_dir())
BASE=ROOT/'experiments/aaai'
os.chdir(ROOT)
from dotenv import load_dotenv as load_env, find_dotenv as find_env
load_env(find_env())
os.chdir(BASE)
if str(BASE) not in sys.path:sys.path.insert(0,str(BASE))
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display,Markdown
from aaai_gait.settings import Settings
from aaai_gait.study_data import build_study_manifest,make_outer_folds,fold_metadata
from aaai_gait.study_runner import prepare_study
from aaai_gait.media import show_video_context,plot_motion
settings=Settings();torch.set_num_threads(2)
fresh_manifest=build_study_manifest(ROOT,settings.pose_dir)
print('Mode:', 'SYNTHETIC SOFTWARE CHECK' if settings.demo else 'REAL GAVD, subject to quality exclusions')
print('Fresh annotations:',len(fresh_manifest),'sequences;',fresh_manifest.source_id.nunique(),'source videos')
video_row=show_video_context(fresh_manifest,frame_number_base=settings.frame_number_base)

## 1. Keep the three kinds of randomness separate

Uniform masking samples hidden positions without reading coordinates, movement speed, condition, or confidence. All sixteen approved joints participate. Observation validity is checked after the mask is drawn, and an invalid target is not replaced by another draw.

Stochastic positional embeddings add noise to position queries. Stochastic latent variables represent alternative completions through a learned distribution. These are different ideas. The primary context experiment uses deterministic positions and no stochastic latent variable so that its information-access comparison is interpretable. The earlier optional positional-noise ablation remains available in Notebook 07. Add latent uncertainty only after checking calibration and diversity against a deterministic comparator.

![Separate sources of stochasticity](images/stochasticity.svg)

# Deduplicated joint whitelist

The selected features use the following 16 distinct BlazePose joints. The table is in numerical order. The model receives only these joints, so its uniform masking sampler cannot mask any other landmark. Original BlazePose indices and compact model positions are different: compact position 0 means BlazePose joint 11.

|BLAZEPOSE_33 index|Keypoint name|Features involved|
|---|---|---|
|11|Left shoulder|Left hip angle; torso scale and orientation; learned shoulder trajectory|
|12|Right shoulder|Right hip angle; torso scale and orientation; learned shoulder trajectory|
|13|Left elbow|Learned arm trajectory and shoulder-elbow-wrist relationship|
|14|Right elbow|Learned arm trajectory and shoulder-elbow-wrist relationship|
|15|Left wrist|Wrist speed, variability and excursion; learned arm swing|
|16|Right wrist|Wrist speed, variability and excursion; learned arm swing|
|23|Left hip|Pelvis center; torso scale; left hip and knee angles; bilateral angle asymmetry|
|24|Right hip|Pelvis center; torso scale; right hip and knee angles; bilateral angle asymmetry|
|25|Left knee|Left hip, knee and ankle angles; bilateral angle asymmetry; learned leg trajectory|
|26|Right knee|Right hip, knee and ankle angles; bilateral angle asymmetry; learned leg trajectory|
|27|Left ankle|Left knee and ankle angles; speed and excursion; ankle separation|
|28|Right ankle|Right knee and ankle angles; speed and excursion; ankle separation|
|29|Left heel|Learned foot orientation and heel trajectory|
|30|Right heel|Learned foot orientation and heel trajectory|
|31|Left foot index|Left ankle angle; learned foot orientation and toe trajectory|
|32|Right foot index|Right ankle angle; learned foot orientation and toe trajectory|

The classical 36-feature baseline does not use every selected coordinate independently. Elbows and heels contribute to learned trajectories in JEPA. Their inclusion is an explicit anatomical design choice. Face and finger-tip landmarks are excluded. The user did not provide a separate list of feature-to-joint definitions, so this table declares the experimental subset rather than claiming it is uniquely required by S-JEPA or clinically optimal.

Bilateral angle differences summarize image-plane asymmetry. They do not identify the affected side or measure three-dimensional joint mechanics. Keep left/right identities, avoid horizontal reflection augmentation in the main analysis, and do not interpret normalized 2D distances as physical stride length.

## 2. Freeze a normal-only encoder, then compare access

Pretrain the skeleton JEPA with uniform masks, an EMA teacher, centered target distributions, and VICReg. Freeze the resulting encoder, teacher, and target center. Each prediction head learns on normal sources only. Its inputs are restricted before any contextual attention.

The own-joint head sees only visible patches of the target joint. The patch-context head also sees other joints in the target four-frame interval. The full-context head sees all visible approved joints. Each target needs at least two visible patches of its own joint.

There are six independent heads: three access conditions for coordinate targets and three for latent targets. Identical initialization within a target type, matching masks, separate optimizers, and fixed update budgets make the comparison easier to interpret. Coordinate MSE and latent KL have different units and must never be subtracted from one another.

![Matched experiment](docs/draft00/images/neuromechanics_experiment.svg)

In [ ]:
from aaai_gait.context_prediction import ContextConfig,context_access,fit_context_models,score_context_models,make_mask_bank
cfg=ContextConfig(pretrain_updates=8,predictor_updates=8,batch_size=8,validation_interval=8,validation_batches=2)
mask=make_mask_bank(cfg,mask_repeats=1)
valid=torch.ones((1,128),dtype=torch.bool)
hidden_ankle_patches=torch.nonzero(mask[0,10::16],as_tuple=True)[0]
target=torch.tensor([int(hidden_ankle_patches[0])*16+10])  # actual hidden left-ankle patch
assert bool(mask[0,target.item()])
access={branch:context_access(mask,valid,target,branch) for branch in ('own','patch','full')}
fig,axes=plt.subplots(1,3,figsize=(12,3))
for ax,(name,allowed) in zip(axes,access.items()):
    ax.imshow(allowed.reshape(8,16),vmin=0,vmax=1,aspect='auto');ax.set_title(name);ax.set_xlabel('Joint slot')
axes[0].set_ylabel('Time patch');plt.tight_layout();plt.show()
assert not (access['own'] & ~access['full']).any()

In [ ]:
x,meta,manifest,ledger=prepare_study(demo=settings.demo,seed=settings.seed)
print('Available QC-passing windows:',x.shape)
display(meta.groupby('label').agg(sources=('source_id','nunique'),sequences=('sequence_id','nunique'),windows=('sequence_id','size')))

In [ ]:
from aaai_gait.predictive_profile import freeze_profile_eligibility
keep,_=freeze_profile_eligibility(x,meta,config=cfg)
a=x[keep];m=meta.loc[keep].reset_index(drop=True)
registry=make_outer_folds(m,seed=settings.seed);m=fold_metadata(m,registry,0)
train=m.split.eq('train').to_numpy();val=m.split.eq('val').to_numpy()
exercise_bundle,history=fit_context_models(a[train],a[val],config=cfg,seed=settings.seed,
    train_source_ids=m.loc[train,'source_id'].to_numpy(),validation_source_ids=m.loc[val,'source_id'].to_numpy(),
    train_labels=m.loc[train,'label'].to_numpy(),validation_labels=m.loc[val,'label'].to_numpy())
print('Eight-update teaching exercise. This is not a converged research model.')
display(pd.DataFrame(history['pretrain']))
display(pd.DataFrame(history['predictors']))

## 3. Inspect matched losses and support

The table records the original BlazePose index, temporal patch, shared mask ID, eligibility, and three losses. A missing target or insufficient own-joint context makes the row ineligible for all three branches. Valid rows with a nonfinite model loss are implementation or optimization failures.

The main profile reads twelve joints in three groups: four left-leg joints, four right-leg joints, and four arm joints. The hips and shoulders still participate in training and masking. They are excluded from the primary score because pelvis centering creates an exact relationship between the hips, which could make the test unusually easy.

In [ ]:
example_rows=score_context_models(exercise_bundle,a[val][:4],mask_repeats=2)
display(example_rows.head(12))
assert np.isfinite(example_rows.loc[example_rows.valid,['loss_own','loss_patch','loss_full']]).all().all()
print(example_rows.groupby('target_type',observed=True).valid.agg(['size','sum']))

## 4. Run the full experiment from a terminal

```bash
uv run python scripts/run_context_study.py --pretrain-updates 512 --predictor-updates 512 --masks 16 --ridge-draws 256
```

This command reads current annotations and poses, freezes eligibility and folds, trains each fold independently, saves compatible checkpoints, and evaluates held-out sources. It also fits a simple source-balanced ridge predictor and an untrained-encoder control. A different corpus, implementation, mask count, or update budget produces a different run directory.

The update count is a budget, not proof of convergence. Inspect validation curves, teacher drift, effective rank, and variance before arguing that representation learning helped. Fix a larger-budget sensitivity using normal validation behavior rather than choosing the run with the best abnormal test score.

A frozen encoder trained with broad context may behave differently when given only one joint. Coordinate targets, the ridge control, and the untrained control help investigate this alternative explanation. Predictive help alone is not a measurement of neural coupling.

## Record what this experiment establishes

Keep the run protocol, source roles, exclusions, training curves, frozen reference parameters, and source scores together. A diagram explains an intended comparison. Only a completed real-data run supplies evidence for it. Read the [implemented protocol](docs/context_study_protocol.md), [current results](docs/context_study_results.md), and [review of the research direction](docs/draft00/README.md) before making a manuscript claim.